In [1]:
# Import required libraries
import json
import jsonlines
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from farasa.segmenter import FarasaSegmenter
from farasa.pos import FarasaPOSTagger
from farasa.ner import FarasaNamedEntityRecognizer
from tqdm.auto import tqdm
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")

In [2]:
generated_by = 'llama-batched'
generation_method = 'by_polishing'

In [3]:
# make sure the notebook is running from the project root
labeled_dataset_path = f'generated_arabic_datasets/{generated_by}/arabic_abstracts_dataset/by_polishing_abstracts_abstracts_generation_filtered.jsonl'

# Load human-AI text pairs
human_texts = []
ai_texts = []
with jsonlines.open(labeled_dataset_path) as reader:
    for obj in reader:
        if 'original_abstract' in obj and 'generated_abstract' in obj:
            # Clean and validate text
            human_text = obj['original_abstract'].strip()
            ai_text = obj['generated_abstract'].strip()
            human_texts.append(human_text)
            ai_texts.append(ai_text)

len(human_texts), len(ai_texts)

(2851, 2851)

In [4]:
def get_text_annotation(annotator, text):
    # annotator should be a function: e.g tagger.tag
    annotated_text = annotator(text).strip('S/S').strip('E/E').strip()
    processed_annotated_text = ''
    for word in annotated_text.split():
        if '/' not in word:
            processed_annotated_text += word + '/UN_TAGGED '
        else:
            processed_annotated_text += f'{word} '
    processed_annotated_text = processed_annotated_text.strip()
    words_tags = []
    for annotated_word in processed_annotated_text.split():
        split_annotated_word = annotated_word.split('/')
        words_tags.append((split_annotated_word[0], split_annotated_word[1]))
    return words_tags

In [5]:
human_texts[0]

'كثيرا ما ارتبطت المصادر التاريخية في الأندلس خاصة منها كتب التراجم والفهرسات والبرامج وغيرها بدراسة حياة العلماء والرواة والقضاة والساسة ؛ وقد تطورت هذه المادة حتى ترك لنا المؤلفون الأندلسيون سلسلة متواصلة الحلقات من كتب التـراجم كالصلة لابن بشكوال ، وصلة الصلة لابن الزبير، والتكملة لكتاب الصلة لابن الآبار، والذيل والتكملة لكتابي الموصول والصلة لابن عبد الملك المراكشي إضافة إلى الإحاطة في أخبار غرناطة لابن الخطيب ، إلا أنها لم تنس أن تشير في ثنايا أو بالأحرى في خواتم هذه المؤلفات إلى فئة المرأة العالمة التي ساهمت في الإنتاج الفكري والحضاري الأندلسي. ومن خلالها سنسعى إلى الوقوف على حالة التعليم عند المرأة الأندلسية ، وكيف كانت تأخذ فنون العلم. وما مدى إسهامها في الفكر التربوي والإنتاج الفكري الأندلسيين ؟.'

In [6]:
ai_texts[0]

'يُقدم هذا البحث دراسة شاملة حول حالة التعليم عند المرأة الأندلسية خلال العصور الوسطى. يُستكشف البحث من خلال سلسلة من المؤلفات التاريخية الأندلسية، مثل "الصلة" لابن بشكوال و"الإحاطة في أخبار غرناطة" لابن الخطيب، الدور الذي لعبته المرأة في الإنتاج الفكري والحضاري الأندلسي. يُسلط البحث الضوء على كيفية تأثر المرأة الأندلسية بالفنون العلمية وتأثيرها في الفكر التربوي الأندلسي. يُقدم البحث تحليلاً مفصلاً حول إسهامات المرأة في الحياة الفكرية والثقافية الأندلسية، مما يوفر رؤية أعمق حول التاريخ الحضاري للمرأة في الأندلس. يُعتبر هذا البحث مساهمة قيمة في فهم تاريخ التعليم عند المرأة في الأندلس وتأثيرها في الفكر التربوي والفكري في ذلك العصر.'

## POS Tagging

In [7]:
annotated_human_texts = []
annotated_ai_texts = []

In [8]:
pos_tagger = FarasaPOSTagger(interactive=True)

[2025-09-02 12:22:48,801 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [9]:
pos_tagger.tag(human_texts[0])

'S/S كثير/NOUN-MS +ا/CASE ما/PART ارتبط +ت/V+PRON ال+ مصادر/DET+NOUN-MS ال+ تاريخي +ة/DET+ADJ+NSUFF-FP في/PREP ال+ أندلس/DET+NOUN-MS خاص +ة/ADJ+NSUFF-FS من/PREP +ها/PRON كتب/NOUN-MS ال+ تراجم/DET+NOUN-MP و+/CONJ ال+ فهرس +ات/DET+NOUN+NSUFF-FD و+/CONJ ال+ برامج/DET+NOUN-MP و+/CONJ غير/PART +ها/PRON ب+/PREP دراس +ة/NOUN+NSUFF-FS حيا +ة/NOUN+NSUFF-FS ال+ علماء/DET+NOUN-MS و+/CONJ ال+ روا +ة/DET+NOUN+NSUFF-MP و+/CONJ ال+ قضا +ة/DET+NOUN+NSUFF-MP و+/CONJ ال+ ساس +ة/DET+NOUN+NSUFF-FP ؛/PUNC و+/CONJ قد/PART تطور +ت/V+PRON هذه/PRON ال+ ماد +ة/DET+NOUN+NSUFF-FS حتى/PREP ترك/NOUN-MS ل+/PREP +نا/PRON ال+ مؤلف +ون/DET+NOUN+NSUFF-MP ال+ أندلسي +ون/DET+ADJ+NSUFF-FD سلسل +ة/NOUN+NSUFF-FS متواصل +ة/ADJ+NSUFF-FP ال+ حلق +ات/DET+NOUN+NSUFF-FP من/PREP كتب/NOUN-MS ال+ تراجم/DET+NOUN-MP ك+/PREP ال+ صل +ة/DET+NOUN+NSUFF-FS ل+/PREP ابن/NOUN-MS بشكوال/NOUN-MS ،/PUNC وصل +ة/NOUN+NSUFF-FS ال+ صل +ة/DET+NOUN+NSUFF-FS ل+/PREP ابن/NOUN-MS ال+ زبير/DET+NOUN-MS ،/PUNC و+/CONJ ال+ تكمل +ة/DET+NOUN+NSUFF-FS ل+/PREP كت

In [10]:
for human_text, ai_text in tqdm(zip(human_texts, ai_texts),total=len(human_texts)):
    annotated_human_texts.extend(get_text_annotation(pos_tagger.tag, human_text))
    annotated_ai_texts.extend(get_text_annotation(pos_tagger.tag, ai_text))

  0%|          | 0/2851 [00:00<?, ?it/s]

In [11]:
human_pos_counts = Counter([tag for word, tag in annotated_human_texts])
ai_pos_counts = Counter([tag for word, tag in annotated_ai_texts])
human_pos_counts, ai_pos_counts

(Counter({'UN_TAGGED': 206387,
          'NOUN-MS': 62493,
          'PREP': 61457,
          'DET+NOUN-MS': 42612,
          'CONJ': 35920,
          'PUNC': 35782,
          'PRON': 34203,
          'V': 24569,
          'PART': 20237,
          'DET+NOUN+NSUFF-FS': 17398,
          'NOUN+NSUFF-FS': 16698,
          'DET+ADJ-MS': 14981,
          'DET+ADJ+NSUFF-FS': 9272,
          'DET+NOUN+NSUFF-FP': 8407,
          'NOUN-FP': 8119,
          'DET+NOUN-MP': 7502,
          'ADJ-MS': 6789,
          'DET+ADJ+NSUFF-FD': 6431,
          'CASE': 6278,
          'V+PRON': 6012,
          'NOUN+NSUFF-FP': 5939,
          'DET+ADJ+NSUFF-FP': 4289,
          'ADJ+NSUFF-FS': 3349,
          'NOUN+NSUFF-FD': 3096,
          'NUM-MP': 2098,
          'ADJ+NSUFF-FP': 1935,
          'NOUN-MP': 1810,
          'DET+NOUN+NSUFF-MP': 1729,
          'ADJ+NSUFF-FD': 1621,
          'DET+ADJ+NSUFF-MP': 1380,
          'PREP+PART': 1363,
          'DET+NOUN+NSUFF-FD': 1307,
          'ADV': 1040,
   

In [12]:
# save counters as json files
with open(f'notebooks/stylometric_analysis/abstracts_dataset/{generated_by}/{generation_method}/human_pos_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        human_pos_counts, 
        f,
        ensure_ascii=False,
        indent=4,
    )
with open(f'notebooks/stylometric_analysis/abstracts_dataset/{generated_by}/{generation_method}/ai_pos_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        ai_pos_counts,
        f,
        ensure_ascii=False,
        indent=4,
    )

## NER

In [13]:
annotated_human_texts = []
annotated_ai_texts = []

In [14]:
ner_tagger = FarasaNamedEntityRecognizer(interactive=True)

[2025-09-02 12:24:44,502 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [15]:
ner_tagger.recognize(human_texts[0])

'كثيرا/O ما/O ارتبطت/O المصادر/O التاريخية/O في/O الأندلس/B-LOC خاصة/O منها/O كتب/O التراجم/O والفهرسات/O والبرامج/O وغيرها/O بدراسة/O حياة/O العلماء/O والرواة/O والقضاة/O والساسة/O ؛/O وقد/O تطورت/O هذه/O المادة/O حتى/O ترك/O لنا/O المؤلفون/O الأندلسيون/O سلسلة/O متواصلة/O الحلقات/O من/O كتب/O التراجم/O كالصلة/O لابن/B-PERS بشكوال/I-PERS ،/O وصلة/O الصلة/O لابن/B-PERS الزبير/I-PERS ،/O والتكملة/O لكتاب/O الصلة/O لابن/B-PERS الآبار/I-PERS ،/O والذيل/O والتكملة/O لكتابي/O الموصول/O والصلة/O لابن/O عبد/B-PERS الملك/I-PERS المراكشي/I-PERS إضافة/O إلى/O الإحاطة/O في/O أخبار/O غرناطة/B-LOC لابن/B-PERS الخطيب/I-PERS ،/O إلا/O أنها/O لم/O تنس/O أن/O تشير/O في/O ثنايا/O أو/O بالأحرى/O في/O خواتم/O هذه/O المؤلفات/O إلى/O فئة/O المرأة/O العالمة/O التي/O ساهمت/O في/O الإنتاج/O الفكري/O والحضاري/O الأندلسي/O ./O ومن/O خلالها/O سنسعى/O إلى/O الوقوف/O على/O حالة/O التعليم/O عند/O المرأة/O الأندلسية/O ،/O وكيف/O كانت/O تأخذ/O فنون/O العلم/O ./O وما/O مدى/O إسهامها/O في/O الفكر/O التربوي/O والإنتاج/O 

In [16]:
for i,(human_text, ai_text) in tqdm(enumerate(zip(human_texts, ai_texts)),total=len(human_texts)):
    annotated_human_texts.extend(get_text_annotation(ner_tagger.recognize, human_text))
    annotated_ai_texts.extend(get_text_annotation(ner_tagger.recognize, ai_text))
    if i == 2230:
        ner_tagger = FarasaNamedEntityRecognizer(interactive=True)

  0%|          | 0/2851 [00:00<?, ?it/s]

[2025-09-02 12:42:56,306 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [17]:
human_ner_counts = Counter([tag for word, tag in annotated_human_texts])
ai_ner_counts = Counter([tag for word, tag in annotated_ai_texts])
human_ner_counts, ai_ner_counts

(Counter({'O': 367269,
          'B-LOC': 2236,
          'I-PERS': 1630,
          'B-PERS': 1345,
          'B-ORG': 403,
          'I-ORG': 401,
          'I-LOC': 249,
          '': 112,
          'UN_TAGGED': 63,
          '02': 17,
          '04': 8,
          '01': 6,
          '03': 6,
          '07': 6,
          '05': 6,
          '12': 5,
          '11': 5,
          '10': 5,
          '16': 4,
          '2019': 4,
          '247': 4,
          '3': 3,
          '1': 3,
          '2018': 3,
          '2017': 3,
          '2016': 2,
          '09': 2,
          '06': 2,
          '131': 2,
          '2014': 1,
          '2003': 1,
          '2007': 1,
          '86': 1,
          '179': 1,
          '54': 1,
          '40': 1,
          '2': 1,
          '13': 1,
          '25': 1,
          '74': 1,
          '172': 1,
          '493': 1,
          '1995': 1,
          '41': 1,
          '8': 1,
          '87': 1,
          '2020': 1}),
 Counter({'O': 313903,
          'B-LO

In [18]:
# save counters as json files
with open(f'notebooks/stylometric_analysis/abstracts_dataset/{generated_by}/{generation_method}/human_ner_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        human_ner_counts, 
        f,
        ensure_ascii=False,
        indent=4,
    )
with open(f'notebooks/stylometric_analysis/abstracts_dataset/{generated_by}/{generation_method}/ai_ner_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        ai_ner_counts,
        f,
        ensure_ascii=False,
        indent=4,
    )

## Segmentation

In [19]:
annotated_human_texts = []
annotated_ai_texts = []

In [20]:
segmenter = FarasaSegmenter(interactive=True)

[2025-09-02 12:48:20,419 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [21]:
segmenter.segment(human_texts[0])

'كثير+ا ما ارتبط+ت ال+مصادر ال+تاريخي+ة في ال+أندلس خاص+ة من+ها كتب ال+تراجم و+ال+فهرس+ات و+ال+برامج و+غير+ها ب+دراس+ة حيا+ة ال+علماء و+ال+روا+ة و+ال+قضا+ة و+ال+ساس+ة ؛ و+قد تطور+ت هذه ال+ماد+ة حتى ترك ل+نا ال+مؤلف+ون ال+أندلسي+ون سلسل+ة متواصل+ة ال+حلق+ات من كتب ال+تراجم ك+ال+صل+ة ل+ابن بشكوال ، وصل+ة ال+صل+ة ل+ابن ال+زبير ، و+ال+تكمل+ة ل+كتاب ال+صل+ة ل+ابن ال+آبار ، و+ال+ذيل و+ال+تكمل+ة ل+كتابي ال+موصول و+ال+صل+ة ل+ابن عبد ال+ملك ال+مراكشي إضاف+ة إلى ال+إحاط+ة في أخبار غرناط+ة ل+ابن ال+خطيب ، إلا أن+ها لم تنس أن تشير في ثنايا أو ب+ال+أحرى في خواتم هذه ال+مؤلف+ات إلى فئ+ة ال+مرأ+ة ال+عالم+ة التي ساهم+ت في ال+إنتاج ال+فكري و+ال+حضاري ال+أندلسي . و+من خلال+ها س+نسعى إلى ال+وقوف على حال+ة ال+تعليم عند ال+مرأ+ة ال+أندلسي+ة ، و+كيف كان+ت تأخذ فنون ال+علم . و+ما مدى إسهام+ها في ال+فكر ال+تربوي و+ال+إنتاج ال+فكري ال+أندلسي+ين ؟ .'

In [22]:
def process_segmentations(annotated_text):
    all_subtokens = list()
    for token in annotated_text.split():
        subtokens = token.split('+')
        subtokens = [subtoken.strip() for subtoken in subtokens]  # noqa: E731
        all_subtokens.extend(subtokens)
    return all_subtokens

In [23]:
human_segmentations = []
ai_segmentations = []

In [24]:
for human_text, ai_text in tqdm(zip(human_texts, ai_texts),total=len(human_texts)):
    human_segmentations.extend(process_segmentations(segmenter.segment(human_text)))
    ai_segmentations.extend(process_segmentations(segmenter.segment(ai_text)))

  0%|          | 0/2851 [00:00<?, ?it/s]

In [25]:
human_segmentations[:10], ai_segmentations[:10]

(['كثير', 'ا', 'ما', 'ارتبط', 'ت', 'ال', 'مصادر', 'ال', 'تاريخي', 'ة'],
 ['يقدم', 'هذا', 'ال', 'بحث', 'دراس', 'ة', 'شامل', 'ة', 'حول', 'حال'])

In [26]:
human_segmentations_counts = Counter(human_segmentations)
ai_segmentations_counts = Counter(ai_segmentations)
human_segmentations_counts.most_common(5), ai_segmentations_counts.most_common(5)

([('ال', 115896), ('ة', 65410), ('و', 32031), ('،', 19519), ('ات', 13280)],
 [('ال', 108443), ('ة', 61820), ('و', 22079), ('،', 16703), ('في', 13890)])

In [27]:
# save counters as json files
with open(f'notebooks/stylometric_analysis/abstracts_dataset/{generated_by}/{generation_method}/human_segmentations_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        human_segmentations_counts, 
        f,
        ensure_ascii=False,
        indent=4,
    )
with open(f'notebooks/stylometric_analysis/abstracts_dataset/{generated_by}/{generation_method}/ai_segmentations_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        ai_segmentations_counts,
        f,
        ensure_ascii=False,
        indent=4,
    )